In [7]:
import pandas as pd
import numpy as np
from collections import Counter
import requests
import regex as re

In [171]:
ebl_dict = pd.read_csv('eBL_Dictionary.csv')

In [172]:
valid_dict = ebl_dict[ebl_dict['definition'].apply(
    lambda x: 
    ((len(re.findall(r'"([^"]*)"',str(x))) == 0)  and
    len(re.findall(r"'[^']*'",str(x))) == 0)
).apply(lambda x: not(x))].reset_index(drop = True)

In [184]:
valid_dict['Meanings'] = valid_dict['definition'].apply(
    lambda x: re.findall(r'"([^"]*)"',str(x)) + re.findall(r"'[^']*'",str(x))
)
valid_dict['Quality'] = 'High'
valid_dict['Source'] = 'Derived from dictionary whose meaning had opened and closed apostrophes('' or ""), meanings is the list of the all strings in those apostrophes'

In [185]:
invalid_dict = ebl_dict[ebl_dict['definition'].apply(
    lambda x: 
    ((len(re.findall(r'"([^"]*)"',str(x))) == 0)  and
    len(re.findall(r"'[^']*'",str(x))) == 0)
).apply(lambda x: (x))].reset_index(drop = True)

In [186]:
invalid_dict_no_brackets = invalid_dict[invalid_dict['definition'].apply(
    lambda x:
    len(re.findall(r'\(([^)]*)\)', str(x))) == 0
)].reset_index(drop = True)

In [187]:
invalid_dict_brackets = invalid_dict[invalid_dict['definition'].apply(
    lambda x:
    len(re.findall(r'\(([^)]*)\)', str(x))) != 0
)].reset_index(drop = True)

In [188]:
invalid_dict_no_brackets = invalid_dict_no_brackets[invalid_dict_no_brackets['definition'].isna().apply(lambda x: not(x))].reset_index(drop = True)

In [189]:
invalid_dict_no_brackets['Quality'] = 'Poor'
invalid_dict_no_brackets['Source'] = 'Derived from dictionary whose meaning did not have any brackets or apostrophes'
invalid_dict_no_brackets['Meanings'] = invalid_dict_no_brackets['definition'].apply(lambda x: [x])

In [190]:
string_to_find = 'cf. GAG'

invalid_dict_brackets['Meanings'] = invalid_dict_brackets[invalid_dict_brackets['definition'].apply(
    lambda x: string_to_find not in str(x)
)].reset_index(drop = True)['definition'].apply(lambda x: re.findall(r'\(([^)]*)\)', str(x)))

invalid_dict_brackets['Source'] = 'Derived from dictionary whose meaning only had brackets, meanings are the strings present in the brackets.'
invalid_dict_brackets['Quality'] = 'Medium'

invalid_dict_brackets = invalid_dict_brackets[invalid_dict_brackets['Meanings'].isna().apply(lambda x: not(x))]

In [191]:
invalid_dict_no_brackets.isna().sum()

word              0
definition        0
derived_from    599
Quality           0
Source            0
Meanings          0
dtype: int64

In [192]:
valid_dict.isna().sum()

word               0
definition         0
derived_from    1868
Meanings           0
Quality            0
Source             0
dtype: int64

In [193]:
final_processed_dict_df = pd.concat([
    valid_dict,
    invalid_dict_brackets,
    invalid_dict_no_brackets
])

In [194]:
final_processed_dict_df.isna().sum()

word               0
definition         0
derived_from    4641
Meanings           0
Quality            0
Source             0
dtype: int64

In [195]:
final_processed_dict_df['word'] = final_processed_dict_df['word'].apply(lambda x: x[:-2])

In [196]:
final_processed_dict_df

,word,definition,derived_from,Meanings,Quality,Source
0,-a,"""my"" (1 sg. pron. suff.)",cf. -ī I,[my],High,Derived from dictionary whose meaning had open...
1,-am,"""to me"" (1 sg. dat. suff.) 2. vent. affix (cf....",NaN,[to me],High,Derived from dictionary whose meaning had open...
2,-iš,"""to; like"" term.-adv. suffix on subst.s and ad...",NaN,[to; like],High,Derived from dictionary whose meaning had open...
3,-ka,"""you"" (2 m. sg. acc. suff.) (cf. GAG p.258b)",NaN,[you],High,Derived from dictionary whose meaning had open...
4,-ki,"""you"" (2 f. sg. acc. suff.) (cf. GAG §42g-k)",NaN,[you],High,Derived from dictionary whose meaning had open...
...,...,...,...,...,...,...
820,ṣêpum,*,NaN,[*],Poor,Derived from dictionary whose meaning did not ...
821,ṭappu,mng. unkn. OB lex.;,ṭapāpu I ?,[mng. unkn. OB lex.;],Poor,Derived from dictionary whose meaning did not ...
822,ṭarāšu,mng. unkn. jB lex. G,NaN,[mng. unkn. jB lex. G],Poor,Derived from dictionary whose meaning did not ...
823,ṭupu,mng. unkn. jB med. in desig. of plant,NaN,[mng. unkn. jB med. in desig. of plant],Poor,Derived from dictionary whose meaning did not ...


In [197]:
final_processed_dict_df.isna().sum()

word               0
definition         0
derived_from    4641
Meanings           0
Quality            0
Source             0
dtype: int64

In [198]:
final_processed_dict_df.to_csv('final_processed_dict_df.csv')